In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override=True)  # override=True 确保 .env 会覆盖系统已有的同名变量
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))  # 确认代理是否生效
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def get_user_info() -> str:
    """Look up information about the current user."""
    return "No user profile on file"

agent = create_agent(
    model = "openrouter:deepseek/deepseek-v4-flash-0731",
    tools = [get_user_info],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable":{"thread_id":"DTEST001"}}
response = agent.invoke(
    {"messages": [{"role":"user","content":"Hi, My name is Bob"}]},
    thread_config
)["messages"][-1].content
print(response)
response = agent.invoke(
    {"messages":[{"role":"user","content":"Hi, What's my name"}]},
    thread_config
)["messages"][-1].content

print(response)

Nice to meet you, Bob! It looks like there isn't a user profile on file for you yet — that's totally okay. 

Is there anything I can help you with today? Whether it's answering a question, working through a problem, or just having a conversation, I'm here to help. What's on your mind?
Based on our conversation, you told me your name is **Bob**! 😊

Note that I don't have a stored user profile on file, so if you'd like me to remember your name going forward, you'll just need to remind me each time we chat. 

Is there anything else I can help you with, Bob?


In [7]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    preferences: dict

agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_user_info],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_id": "user_123",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}})

print(result)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='dfa80f1a-3485-4e22-a490-34d7d5dc2716'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user greeted me. I should respond in a friendly way. Since they mentioned the user, maybe I should look up user info? Actually, the greeting doesn\'t require it. But let me consider - the tools include get_user_info. The user just said "Hello". I can greet them back. But maybe it\'s useful to look up user info to personalize. Let me do that since I have the tool available and it could help personalize the response.\n\nActually, on a simple greeting, it\'s fine to just greet back. But let me consider looking up user info to personalize. I\'ll call get_user_info to greet them properly.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user greeted me. I should respond in a friendly way. Since they mentioned the user, maybe I should look up user i

In [10]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str,Any] | None:
    """Keep only the last few messages to fit context window"""
    messages = state["messages"]
    if len(messages) <= 3:
        return None
    
    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages":[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }
agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob! You told me at the very beginning of our chat. 😊

Would you like another poem, or is there something else you'd like to chat about, Bob?


In [12]:
# 删除置顶的消息
from langchain.messages import RemoveMessage
def delete_messages(state):
    messages = state["messages"]
    if len(messages) > 2:
        return {"messages":[RemoveMessage(id=m.id) for m in messages[:2]]}

# 删除全部消息
from langgraph.graph.message import REMOVE_ALL_MESSAGES
def delete_messages(state):
    return {"messages":[RemoveMessage(id=REMOVE_ALL_MESSAGES)]}


In [ ]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent,AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from requests import delete

@after_model
def delete_old_messages(state:AgentState,runtime:Runtime)->dict | None:
    """Remove old messages to keep conversation manageabled."""
    messages = state["messages"]
    if len(messages)>2:
        # remove messages
        return {"messages":{RemoveMessage(id=m.id) for m in messages[:2]}}

agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    system_prompt="Please be concise and to the point.",
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}


e:\code2\d-api\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
e:\code2\d-api\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


[('human', "hi! I'm bob")]
[('human', "hi! I'm bob"), ('ai', [{'type': 'reasoning', 'reasoning': '1.  **Analyze the User\'s Input**:\n    *   Input: "hi! I\'m bob"\n    *   Intent: Greeting and self-introduction. The user\'s name is Bob.\n    *   Constraint: "Please be concise and to the point." (Keep the response short, friendly, and acknowledge the introduction).\n\n2.  **Determine the appropriate response**:\n    *   Acknowledge the greeting.\n    *   Acknowledge the name.\n    *   Offer assistance or ask what they need, keeping it brief.\n\n3.  **Drafting the response**:\n    *   *Option 1 (Too wordy)*: "Hello Bob, it is a pleasure to meet you. How may I be of service to you today? I am here to assist you with any questions or tasks you might have."\n    *   *Option 2 (Concise)*: "Hi Bob! What can I help you with today?" - This is very concise, but perhaps I should acknowledge the name clearly.\n    *   *Option 3 (Good balance)*: "Hey Bob! Nice to meet you. What can I do for you?" 

TypeError: unhashable type: 'RemoveMessage'